In [ ]:
# 2. Import libraries
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print("✅ Libraries imported!")

print("=== LOADING BEST MODEL ===")

try:
    model_path = '/content'
    model = tf.keras.models.load_model(model_path)
    print(f"✅ Loaded: (69.99% val)")
except:
    try:
        model_path = '/content'
        model = tf.keras.models.load_model(model_path)
        print("✅ Loaded: FINAL_NIGHT_MODEL.keras")
    except:
        model_path = '/content'
        model = tf.keras.models.load_model(model_path)
        print("✅ Loaded: (66% balanced)")

print(f"🎯 Model loaded successfully from: {model_path}")

In [ ]:
def check_dataset_structure():
    train_path = "/content"
    val_path = "/content"
    test_path = "/content"

    print("=== CHECKING DATASET STRUCTURE ===")
    for split_path, split_name in [(train_path, 'TRAIN'), (val_path, 'VAL'), (test_path, 'TEST')]:
        print(f"\n{split_name} SET:")
        if os.path.exists(split_path):
            for class_name in os.listdir(split_path):
                class_path = os.path.join(split_path, class_name)
                if os.path.isdir(class_path):
                    num_images = len([f for f in os.listdir(class_path) if f.endswith(('.jpg', '.png', '.jpeg'))])
                    print(f"  {class_name}: {num_images} images")
        else:
            print(f"  Path not found: {split_path}")

check_dataset_structure()

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_path = ""
val_path = ""
test_path = ""

print(f"Train path: {train_path}")
print(f"Val path: {val_path}")
print(f"Test path: {test_path}")

In [ ]:
# === LOADING WITH FOCAL LOSS FIX ===
print("=== LOADING WITH FOCAL LOSS FIX ===")

def focal_loss(gamma=2.0, alpha=0.25):
    def focal_loss_fixed(y_true, y_pred):
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)
        cross_entropy = -y_true * tf.math.log(y_pred)
        weight = alpha * y_true * tf.pow((1 - y_pred), gamma)
        focal_loss_value = weight * cross_entropy
        return tf.reduce_mean(focal_loss_value, axis=-1)
    return focal_loss_fixed

custom_objects = {
    'focal_loss_fixed': focal_loss(),
    'focal_loss': focal_loss
}

model = tf.keras.models.load_model(
    '/content',
    custom_objects=custom_objects
)
print("✅ SUCCESS! Loaded with focal loss fix!")

# === SETUP TEST GENERATOR ===
from tensorflow.keras.preprocessing.image import ImageDataGenerator

test_path = "/content"
IMG_HEIGHT, IMG_WIDTH = model.input_shape[1], model.input_shape[2]

test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_directory(
    directory=test_path,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=16,
    class_mode='categorical',
    shuffle=False
)

print("✅ Test generator ready!")

test_loss, test_accuracy = model.evaluate(test_generator)
print(f"🎯 LAST NIGHT MODEL ACCURACY: {test_accuracy*100:.2f}%")

In [ ]:
print("=== TESTING ON ACTUAL IMAGES ===")

def test_single_image(image_path, model, class_names, img_size=(384, 384)):
    # Load and preprocess
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=img_size)
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0) / 255.0

    # Predict
    predictions = model.predict(img_array, verbose=0)
    predicted_class = class_names[np.argmax(predictions[0])]
    confidence = np.max(predictions[0])

    # Display
    plt.figure(figsize=(10, 6))
    plt.imshow(img)
    plt.title(f"Predicted: {predicted_class}\nConfidence: {confidence:.2%}",
              fontsize=16, weight='bold', pad=20)
    plt.axis('off')
    plt.show()

    # Show probabilities
    print(f"📊 PREDICTION RESULTS:")
    for i, class_name in enumerate(class_names):
        prob = predictions[0][i]
        print(f"   {class_name}: {prob:.3f} ({prob*100:.1f}%)")

    return predicted_class, confidence

# Get class names
class_names = list(test_generator.class_indices.keys())
print(f"🎯 Class names: {class_names}")

import os

test_images = []
for class_name in class_names:
    class_test_path = f"/content/{class_name}"
    if os.path.exists(class_test_path):
        images = os.listdir(class_test_path)
        if images:
            test_images.append(f"{class_test_path}/{images[0]}")
            print(f"📁 Found {class_name} image: {images[0]}")


if test_images:
    print("\n🧪 TESTING ON FOUND IMAGES:")
    for img_path in test_images:
        print(f"\n🔍 Testing: {img_path}")
        test_single_image(img_path, model, class_names)
else:
    print("\n❌ No test images found automatically.")
    print("💡 Manual testing: Use test_single_image('your/image/path.jpg', model, class_names)")

In [ ]:
print("=== SMART ENSEMBLE FOR 80% TARGET ===")

def focal_loss(gamma=2.0, alpha=0.25):
    def focal_loss_fixed(y_true, y_pred):
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)
        cross_entropy = -y_true * tf.math.log(y_pred)
        weight = alpha * y_true * tf.pow((1 - y_pred), gamma)
        focal_loss_value = weight * cross_entropy
        return tf.reduce_mean(focal_loss_value, axis=-1)
    return focal_loss_fixed


def load_all_models():
    models = []
    model_files = [
        '/content'
    ]

    custom_objects = {'focal_loss_fixed': focal_loss(), 'focal_loss': focal_loss}

    for model_path in model_files:
        try:
            if 'FINAL_NIGHT' in model_path:
                model = tf.keras.models.load_model(model_path, custom_objects=custom_objects)
            else:
                model = tf.keras.models.load_model(model_path)
            models.append(model)
            print(f"✅ Loaded: {os.path.basename(model_path)}")
        except Exception as e:
            print(f"❌ Failed {os.path.basename(model_path)}: {str(e)[:100]}...")

    return models

models = load_all_models()
print(f"🎯 Ensemble size: {len(models)} models")

# SMART ENSEMBLE with confidence weighting
def smart_ensemble_predict(image_path, models, class_names, img_size=(384, 384)):

    img = tf.keras.preprocessing.image.load_img(image_path, target_size=img_size)
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0) / 255.0


    all_predictions = []
    model_confidences = []

    for i, model in enumerate(models):
        prediction = model.predict(img_array, verbose=0)[0]
        predicted_class_idx = np.argmax(prediction)
        confidence = prediction[predicted_class_idx]

        all_predictions.append(prediction)
        model_confidences.append(confidence)

        print(f"🤖 Model {i+1}: {class_names[predicted_class_idx]} ({confidence:.2%})")

    weighted_predictions = np.zeros_like(all_predictions[0])
    total_confidence = sum(model_confidences)

    for i, pred in enumerate(all_predictions):
        weight = model_confidences[i] / total_confidence
        weighted_predictions += pred * weight

    final_prediction = weighted_predictions
    predicted_class = class_names[np.argmax(final_prediction)]
    final_confidence = np.max(final_prediction)

    # Visualization
    plt.figure(figsize=(15, 5))

    # Original image
    plt.subplot(1, 3, 1)
    plt.imshow(img)
    plt.title("Input Image", fontsize=14, weight='bold')
    plt.axis('off')

    # Model predictions
    plt.subplot(1, 3, 2)
    model_names = [f"Model {i+1}" for i in range(len(models))]
    plt.bar(model_names, model_confidences, color=['blue', 'green', 'orange'])
    plt.title("Individual Model Confidences", fontsize=12)
    plt.xticks(rotation=45)
    plt.ylim(0, 1)

    # Final ensemble probabilities
    plt.subplot(1, 3, 3)
    colors = ['red', 'blue', 'green']
    bars = plt.bar(class_names, final_prediction, color=colors, alpha=0.7)
    plt.title(f"Ensemble Result\n{predicted_class} ({final_confidence:.2%})",
              fontsize=14, weight='bold')
    plt.ylim(0, 1)

    # Add probability labels
    for bar, prob in zip(bars, final_prediction):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{prob:.3f}', ha='center', va='bottom', fontweight='bold')

    plt.tight_layout()
    plt.show()

    print(f"\n🎯 SMART ENSEMBLE RESULT: {predicted_class} ({final_confidence:.2%})")
    return predicted_class, final_confidence

# Get class names
class_names = list(test_generator.class_indices.keys())
print(f"🎯 Class names: {class_names}")

# Test the smart ensemble
print("\n🧪 TESTING SMART ENSEMBLE:")

# Find test images automatically
test_images = []
for class_name in class_names:
    class_test_path = f"/content/{class_name}"
    if os.path.exists(class_test_path):
        images = os.listdir(class_test_path)
        if images:
            test_images.append(f"{class_test_path}/{images[0]}")
            print(f"📁 Found {class_name} image: {images[0]}")

# Test on found images
if test_images:
    for img_path in test_images:
        print(f"\n" + "="*60)
        print(f"🔍 Testing: {os.path.basename(img_path)}")
        smart_ensemble_predict(img_path, models, class_names)
else:
    print("❌ No test images found. Please check your file paths.")

In [ ]:
print("=== BALANCED GAN vs REAL SOLUTION ===")

from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_path = "/content"
val_path = "/content"
test_path = "/content"
# Data generators
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)

# Create generators
train_generator = train_datagen.flow_from_directory(
    directory=train_path,
    target_size=(384, 384),
    batch_size=16,
    class_mode='categorical',
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    directory=val_path,
    target_size=(384, 384),
    batch_size=16,
    class_mode='categorical',
    shuffle=False
)

print("✅ Data generators ready!")

# Strategy: Targeted data augmentation + balanced training
def create_balanced_gan_fix(model):

    model.trainable = True

    # Gradual unfreezing - only last 40 layers
    for layer in model.layers[:-40]:
        layer.trainable = False

    # BALANCED weights
    balanced_weights = {
        0: 1.3,   # GAN: +30% focus (but not extreme)
        1: 1.0,   # Graphics: Neutral
        2: 1.0    # Real: Neutral
    }

    print(f"🎯 Balanced weights: {balanced_weights}")
    print("Strategy: Moderate GAN focus without punishing other classes")


    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-6),  # Very low LR
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model, balanced_weights

balanced_model, balanced_weights = create_balanced_gan_fix(models[0])

# SMART CALLBACKS to maintain balance
balanced_callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=3,
        restore_best_weights=True,
        min_delta=0.001
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        verbose=1
    )
]

print("🚀 Starting balanced training...")
balanced_history = balanced_model.fit(
    train_generator,
    epochs=6,  # More epochs but with early stopping
    validation_data=val_generator,
    class_weight=balanced_weights,
    callbacks=balanced_callbacks,
    verbose=1
)

print("✅ Balanced training completed!")